# DeepTrace — Phases 5 & 6: robustness, explainability, export, model selection

**Run on:** Kaggle (free) · GPU on · Internet **On**
**Inputs:** `deeptrace-processed`, the `140k Real and Fake Faces` dataset (raw images for robustness), and
every `deeptrace-runs-<model>` dataset from `02_train_and_evaluate`.

1. **Robustness** — JPEG 100→30, downscaling, blur, screenshot chains, through the serving pipeline.
2. **Explainability** — Grad-CAM / Grad-CAM++ (CNN), Grad-CAM + attention rollout (ViT), worst-error
   galleries, and the model-randomisation sanity check.
3. **FFT analysis** of real vs fake crops.
4. **Export** each model to ONNX (fails loudly on any parity mismatch) and **benchmark CPU latency**.
5. **Select** the deployment model with the pre-registered rule and render `docs/METRICS.md`.

In [ ]:
# ---- Setup: clone your repo + install the few extras Kaggle doesn't ship ----
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/Vuday3336/DeepTrace-AI-face-detector.git"
ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path("/content")
REPO = WORK / "deeptrace"
ML = REPO / "ml"

if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
os.chdir(ML)
print("Working in", Path.cwd())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements/kaggle.txt", "-r", "requirements/train.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements/facenet.txt", "--no-deps"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], check=True)
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"

import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

def run(*args):
    print("$", " ".join(map(str, args)))
    subprocess.run([sys.executable, *map(str, args)], check=True)

In [ ]:
# ---- Locate Phase 2 outputs among the attached inputs ----
import os, json, shutil
from pathlib import Path

def find_dir(root, predicate, max_depth=6):
    base = len(Path(root).parts)
    for dirpath, dirnames, _ in os.walk(root):
        p = Path(dirpath)
        if len(p.parts) - base >= max_depth:
            dirnames.clear(); continue
        if predicate(p):
            return p
    return None

INPUT = Path("/kaggle/input") if ON_KAGGLE else Path("/content/input")
PROCESSED = find_dir(INPUT, lambda p: p.name == "processed" and (p / "140k").is_dir())
MANIFESTS = find_dir(INPUT, lambda p: (p / "140k_processed.csv.gz").is_file())
if PROCESSED is None or MANIFESTS is None:
    raise FileNotFoundError("Attach the 'deeptrace-processed' dataset (output of 01_data_prep_and_audit).")
RAW_140K = find_dir(INPUT, lambda p: p.name == "real-vs-fake" and (p / "test").is_dir())
RAW_CROSSGEN = find_dir(INPUT, lambda p: p.name == "crossgen" and (p / "fake").is_dir())
PROCESSED_MANIFESTS = [str(m) for m in (MANIFESTS / "140k_processed.csv.gz", MANIFESTS / "crossgen_processed.csv.gz") if m.is_file()]
print("processed crops:", PROCESSED)
print("manifests:", PROCESSED_MANIFESTS)
print("raw 140k:", RAW_140K, "| raw crossgen:", RAW_CROSSGEN)

In [ ]:
CHECKPOINTS = {}
REPORTS = ML / "reports"
for model in ("effnet_b0", "convnext_tiny", "vit_small", "clip_probe"):
    found = find_dir(INPUT, lambda p, m=model: p.name.startswith(f"{m}_seed") and (p / "best.pt").is_file())
    reports = find_dir(INPUT, lambda p, m=model: (p / f"calibration_{m}.json").is_file())
    if found and reports:
        CHECKPOINTS[model] = found / "best.pt"
        for f in reports.glob(f"*_{model}.*"):
            shutil.copy(f, REPORTS / f.name)
print("models found:", {k: str(v) for k, v in CHECKPOINTS.items()})
VERSION = "1.0.0"

## 1. Robustness

In [ ]:
roots = ["--root", f"140k={RAW_140K}"] + (["--root", f"crossgen={RAW_CROSSGEN}"] if RAW_CROSSGEN else [])
pm_args = [a for m in PROCESSED_MANIFESTS for a in ("--processed-manifest", m)]
for model, ckpt in CHECKPOINTS.items():
    run("scripts/robustness.py", "--checkpoint", ckpt, "--model-name", model, *pm_args, *roots,
        "--calibration", REPORTS / f"calibration_{model}.json")

## 2. Explainability galleries + sanity check

In [ ]:
from IPython.display import Image as ShowImage, display
for model, ckpt in CHECKPOINTS.items():
    run("scripts/explain.py", "--checkpoint", ckpt, "--model-name", model,
        "--predictions", REPORTS / f"predictions_{model}.csv", "--calibration", REPORTS / f"calibration_{model}.json",
        "--data-root", PROCESSED, "--device", "cuda" if torch.cuda.is_available() else "cpu")
    print(model, json.load(open(REPORTS / f"explain_{model}.json")).get("sanity_check"))
    for png in sorted((REPO / f"docs/plots/explain/{model}").glob("*.png")):
        print(png.name); display(ShowImage(filename=str(png)))

**Write your interpretation in `docs/EXPLAINABILITY.md`** as observations, not claims about what the model
"understands" (e.g. *"in 8 sampled false positives on crossgen, heat concentrates on hair edges"*). If the
sanity-check correlation is high, state that the heatmaps are unreliable for that model.

## 3. Frequency analysis

In [ ]:
run("scripts/fft_analysis.py", *[a for m in PROCESSED_MANIFESTS for a in ("--processed-manifest", m)],
    "--data-root", PROCESSED)
for name in ("radial_profiles", "difference_vs_reference"):
    f = REPO / f"docs/plots/fft/{name}.png"
    if f.exists():
        display(ShowImage(filename=str(f)))

## 4. Export to ONNX + CPU latency

In [ ]:
ARTIFACTS = WORK / "artifacts"
for model, ckpt in CHECKPOINTS.items():
    out = ARTIFACTS / model
    run("scripts/export_model.py", "--checkpoint", ckpt, "--calibration", REPORTS / f"calibration_{model}.json",
        "--eval-report", REPORTS / f"eval_{model}.json", "--robustness-report", REPORTS / f"robustness_{model}.json",
        "--version", VERSION, "--out-dir", out)
    # 2 threads approximates the free Hugging Face Spaces CPU; Kaggle CPUs differ, so treat as indicative.
    run("scripts/benchmark_latency.py", "--bundle-dir", out, "--threads", "2", "--checkpoint", ckpt)

## 5. Model selection (pre-registered rule) + metrics report

In [ ]:
evals = [a for m in CHECKPOINTS for a in ("--eval", REPORTS / f"eval_{m}.json")]
lats = [a for m in CHECKPOINTS for a in ("--latency", REPORTS / f"latency_{m}.json")]
run("scripts/compare_models.py", *evals, *lats)
print(json.dumps(json.load(open(REPORTS / "model_selection.json"))["decision"], indent=2))
run("scripts/render_metrics_md.py")
print(open(REPO / "docs/METRICS.md").read()[:3000])

## Save & publish

1. Commit the run (**Save & Run All**), download `deeptrace_final_artifacts.zip`, and copy `ml/reports`,
   `docs/plots` and `docs/METRICS.md` into your repo.
2. Upload the **selected** bundle to the Hugging Face Hub (free), locally or with a Kaggle secret `HF_TOKEN`:
   `python scripts/upload_model_to_hub.py --bundle-dir artifacts/<winner> --repo-id <user>/deeptrace-model`

In [ ]:
bundle = WORK / "deeptrace_final_artifacts"
shutil.rmtree(bundle, ignore_errors=True)
shutil.copytree(REPORTS, bundle / "ml/reports")
shutil.copytree(REPO / "docs/plots", bundle / "docs/plots", dirs_exist_ok=True)
shutil.copy(REPO / "docs/METRICS.md", bundle / "docs/METRICS.md")
shutil.copytree(ARTIFACTS, bundle / "artifacts")
print(shutil.make_archive(str(bundle), "zip", bundle))